In [16]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import os

In [13]:
notebook_dir = os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
data_dir = os.path.join(parent_dir, 'data')
data_processed_dir = os.path.join(data_dir, 'processed')
print(notebook_dir)
print(parent_dir)
print(data_dir)

c:\Users\minhtam.nguyen\0_Data_project\demokit-megaverse\notebook
c:\Users\minhtam.nguyen\0_Data_project\demokit-megaverse
c:\Users\minhtam.nguyen\0_Data_project\demokit-megaverse\data


In [ ]:
file_dir = os.path.join(data_processed_dir, 'z_test_product_master_data_categorized.csv')
df = pd.read_csv(file_dir)
df.head()

,product_id,product_title,mrp,price,product_tags,product_url,product_category,category_cf_score,combo_set,product_function,function_cf_score
0,781070,Olay Ultra Lightweight Moisturiser: Luminous W...,1999,1599,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...,face,0.635290,NaN,skin care,0.866548
1,739418,Olay Regenerist Whip Mini and Ultimate Eye Cre...,2198,1943,NaN,https://www.nykaa.com/olay-regenerist-whip-min...,eyes,0.683865,NaN,skin care,0.577126
2,6836720,"Olay Aha & Niacinamide Super Cream , Acne Mark...",1999,1399,FEATURED,https://www.nykaa.com/olay-aha-niacinamide-sup...,face,0.799516,NaN,skin care,0.941370
3,61305,Olay White Radiance Day & Night Cream for Brig...,2798,2378,NaN,https://www.nykaa.com/olay-white-radiance-day-...,face,0.725855,NaN,skin care,0.889267
4,60541,Olay Total Effects 7 In One Anti-Ageing Day Cr...,375,375,NaN,https://www.nykaa.com/olay-total-effects-7-in-...,face,0.752202,NaN,skin care,0.908070
...,...,...,...,...,...,...,...,...,...,...,...
290,1135934,Herbal Essences Argan 2 Shampoo + Conditioner,1850,1295,NaN,https://www.nykaa.com/herbal-essences-argan-2-...,hair,0.477264,NaN,skin care,0.946020
291,1044492,Herbal Essences Aloe & Bamboo Shampoo For Soft...,750,525,FEATURED,https://www.nykaa.com/herbal-essences-potent-a...,hair,0.806113,NaN,skin care,0.805634
292,1044491,Herbal Essences Aloe & Bamboo Conditioner Soft...,825,578,NaN,https://www.nykaa.com/herbal-essences-potent-a...,hair,0.811485,NaN,skin care,0.642279
293,1044490,Herbal Essences Aloe & Eucalyptus Shampoo For ...,750,525,FEATURED,https://www.nykaa.com/herbal-essences-potent-a...,hair,0.841139,NaN,skin care,0.764185


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_id         295 non-null    int64  
 1   product_title      295 non-null    object 
 2   mrp                295 non-null    int64  
 3   price              295 non-null    int64  
 4   product_tags       30 non-null     object 
 5   product_url        295 non-null    object 
 6   product_category   295 non-null    object 
 7   category_cf_score  295 non-null    float64
 8   combo_set          45 non-null     object 
 9   product_function   295 non-null    object 
 10  function_cf_score  295 non-null    float64
dtypes: float64(2), int64(3), object(6)
memory usage: 25.5+ KB


In [37]:
schema = pa.schema([
    ('product_id', pa.int64()),           
    ('product_title', pa.string()),       
    ('mrp', pa.int64()),                
    ('price', pa.int64()),              
    ('product_tags', pa.string()), 
    ('product_url', pa.string()),         
    ('product_category', pa.string()),    
    ('category_cf_score', pa.float64()),  
    ('combo_set', pa.string()),            
    ('product_function', pa.string()),    
    ('function_cf_score', pa.float64())   
])
# 4. Convert and Save
table = pa.Table.from_pandas(df, schema=schema)

pq.write_table(table, os.path.join(data_processed_dir,'product_master_data_categorized.parquet'))

print("Conversion complete: product_data_fixed.parquet created.")

Conversion complete: product_data_fixed.parquet created.


DROP TABLE z_test_dim_product;

CREATE EXTERNAL TABLE IF NOT EXISTS `z_test_dim_product` (
  `product_id` bigint,
  `product_title` string,
  `mrp` bigint,
  `price` bigint,
  `product_tags` string,
  `product_url` string,
  `product_category` string,
  `category_cf_score` double,
  `combo_set` string,
  `product_function` string,
  `function_cf_score` double
)
STORED AS PARQUET
LOCATION 's3://offy-bi-demo/z_test/test_product/'
TBLPROPERTIES ('classification'='parquet');

# Upload to S3

In [30]:
import sys
parent_dir = os.path.abspath("..")
print(parent_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

# 2. Get the absolute path to 'src' itself
src_path = os.path.join(parent_dir, "src")

# 3. Add BOTH to the top of sys.path
for p in [parent_dir, src_path]:
    if p not in sys.path:
        sys.path.insert(0, p)
print(src_path)
# push_data_to_s3

c:\Users\minhtam.nguyen\0_Data_project\demokit-megaverse
c:\Users\minhtam.nguyen\0_Data_project\demokit-megaverse\src


In [57]:
from src.loader.load_function import push_data_to_s3

In [58]:
local_path = os.path.join(data_processed_dir,'product_master_data_categorized.parquet')
s3_key = 'z_test/test_product/product_master_data_categorized.parquet'
push_data_to_s3(local_path, s3_key)

False

In [48]:
import src.config as config
import importlib
importlib.reload(config) # Forces Python to re-read the config.py file
print(f"CURRENT BUCKET: {config.S3_BUCKET_NAME}")

CURRENT BUCKET: offy-bi-demo
